In [1]:
import sys, os, subprocess
from pathlib import Path

# Colab: clone repo and install deps. Local: resolve root from CWD.
try:
    import google.colab  # noqa
    REPO = '/content/Katabatic'
    if not os.path.exists(REPO):
        subprocess.run(
            ['git', 'clone', 'https://github.com/lukebrumby/katabatic-personal.git', REPO],
            check=True
        )
    os.chdir(REPO)
    sys.path.insert(0, REPO)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
    ROOT = Path(REPO)
except ImportError:
    ROOT = Path.cwd().resolve()
    for _ in range(5):
        if (ROOT / 'pyproject.toml').exists() or (ROOT / 'raw_data').exists():
            break
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models_luke.tvae.models import TVAEModel

ROOT: /content/Katabatic


In [2]:
MODEL = lambda: TVAEModel(
    embedding_dim=128,
    compress_dims=(128, 128),
    decompress_dims=(128, 128),
    l2scale=1e-5,
    batch_size=500,
    epochs=300,
    loss_factor=2,
    cuda=True,
)

In [3]:
DATASETS = ["car", "adult", "magic", "shuttle", "nursery"]

for dataset in DATASETS:
    dataset_path = ROOT / "raw_data" / f"{dataset}.csv"
    output_path = ROOT / "discretized_data" / f"{dataset}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Preprocessing {dataset}...")
    discretize_preprocess(str(dataset_path), str(output_path))

Preprocessing car...
Preprocessing: /content/Katabatic/raw_data/car.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/car.csv
Preprocessing adult...
Preprocessing: /content/Katabatic/raw_data/adult.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/adult.csv
Preprocessing magic...
Preprocessing: /content/Katabatic/raw_data/magic.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/magic.csv
Preprocessing shuttle...
Preprocessing: /content/Katabatic/raw_data/shuttle.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/shuttle.csv
Preprocessing nursery...
Preprocessing: /content/Katabatic/raw_data/nursery.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/nursery.csv


In [4]:
for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"TVAE -> {dataset}")
    input_csv = str(ROOT / "discretized_data" / f"{dataset}.csv")
    output_dir = str(ROOT / "sample_data" / dataset)
    synthetic_dir = str(ROOT / "synthetic" / dataset / "tvae")

    pipeline = TrainTestSplitPipeline(model=MODEL)
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=output_dir,
    )
    print(result)


TVAE -> car
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
[TVAE] Detected discrete columns: ['0', '1', '2', '3', '4', '5', '6']
[TVAE] Creating metadata...
[TVAE] Initializing TVAE with 300 epochs...
[TVAE] Training on 1382 samples...
[TVAE] Finished training in 36.11 seconds.
[TVAE] Generating 1382 synthetic samples...
[TVAE] Synthetic data saved:
  X -> /content/Katabatic/synthetic/car/tvae/x_synth.csv
  y -> /content/Katabatic/synthetic/car/tvae/y_synth.csv

Results saved to: Results/car/tvae_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6994
F1 Score: 0.6359

MLP:
Accuracy: 0.8844
F1 Score: 0.8629

RF:
Accuracy

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [0 1 3]